# Draft night — Sleeper, 14-team superflex, half-PPR

Thursday 3 September 2026, 18:30. Snake, no third-round reversal, 15 rounds, 120-second pick timer,
**seat 1**. Picks: **1, 28, 29, 56, 57, 84, 85, 112, 113, 140, 141, 168, 169, 196, 197** — one pick
at the top and then seven back-to-back pairs.

Everything below is computed from the warehouse, not typed in. Re-run after a rebuild and the
findings update; if a number here moved, the draft moved.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.query import q

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

LEAGUE = "sleeper"
MY_SEAT = 1
MY_PICKS = [1, 28, 29, 56, 57, 84, 85, 112, 113, 140, 141, 168, 169, 196, 197]

## Why a normal ADP board is the wrong sheet

Superflex does not change how anyone scores. It changes how many quarterbacks have a job — from 14
to 28 in this league — and that reprices the entire position. The same player, two markets:

In [2]:
q('''
    SELECT player_name, position,
           MAX(consensus_adp) FILTER (WHERE format = '1qb')       AS adp_1qb,
           MAX(consensus_adp) FILTER (WHERE format = 'superflex') AS adp_superflex
    FROM adp_consensus
    WHERE season = 2026 AND position = 'QB'
    GROUP BY 1, 2
    HAVING MAX(consensus_adp) FILTER (WHERE format = 'superflex') IS NOT NULL
    -- player_name breaks ties: Hurts and Stafford share a superflex ADP, and an unstable
    -- sort there makes a no-op re-run produce a diff.
    ORDER BY adp_superflex, player_name
    LIMIT 10
''')

,player_name,position,adp_1qb,adp_superflex
0,Josh Allen,QB,27.60,1.8
1,Drake Maye,QB,51.80,7.4
2,Lamar Jackson,QB,47.20,8.5
3,Joe Burrow,QB,54.70,9.5
4,Dak Prescott,QB,73.10,13.8
5,Jayden Daniels,QB,65.55,14.1
6,Matthew Stafford,QB,90.65,17.2
7,Jalen Hurts,QB,69.85,17.3
8,Brock Purdy,QB,96.65,19.7
9,Trevor Lawrence,QB,88.60,19.8


Josh Allen is the 1.01 of one market and an afterthought in the other. Any model reading the 1QB
board for this league is not slightly wrong — it is pricing a different game.

## Replacement level is why

A player is worth his projection **minus what the position costs for free**: the last player who
still has to start somewhere once all 14 teams fill their slots. Count the jobs and the whole board
falls out of it.

In [3]:
q('''
    SELECT DISTINCT position, starters_at_position AS jobs,
           round(replacement_level_points, 1) AS replacement_level
    FROM draft_board
    WHERE league_key = ? AND position IN ('QB','RB','WR','TE')
    ORDER BY position
''', [LEAGUE])

,position,jobs,replacement_level
0,QB,28,232.8
1,RB,31,133.1
2,TE,14,117.0
3,WR,39,138.3


28 quarterback jobs against roughly 32 startable NFL quarterbacks. That is the whole story: the
replacement quarterback is a backup, so an elite one is worth far more here than in a 1QB league,
where replacement is QB15 and perfectly startable.

## The board

Ranked on durability-adjusted points over replacement. `ol_tier` flags running backs behind a weak
offensive line — see below for why it is a warning rather than a ranking input.

In [4]:
q('''
    SELECT position AS pos, player_name, team, consensus_adp AS adp,
           round(points_over_replacement, 1) AS por,
           round(projected_points_adjusted, 1) AS proj,
           round(availability, 3) AS avail,
           round(projected_floor, 0) AS floor, round(projected_ceiling, 0) AS ceil,
           ol_tier
    FROM draft_board
    WHERE league_key = ? AND consensus_adp IS NOT NULL
    ORDER BY points_over_replacement DESC
    LIMIT 40
''', [LEAGUE])

,pos,player_name,team,adp,por,proj,avail,floor,ceil,ol_tier
0,RB,Jahmyr Gibbs,DET,1.8,181.7,314.7,0.950,321.0,343.0,Q3
1,RB,Bijan Robinson,ATL,2.6,172.2,305.3,0.970,307.0,327.0,Q4 best
2,QB,Josh Allen,BUF,1.8,145.7,378.5,0.985,367.0,396.0,None
3,RB,Jonathan Taylor,IND,5.5,132.4,265.5,0.920,261.0,302.0,Q4 best
4,WR,Ja'Marr Chase,CIN,6.4,130.0,268.3,0.972,269.0,278.0,None
5,WR,Puka Nacua,LA,4.7,128.1,266.3,0.920,275.0,296.0,None
6,RB,De'Von Achane,MIA,20.5,122.4,255.4,0.962,260.0,276.0,Q2
7,WR,Jaxon Smith-Njigba,SEA,8.3,117.5,255.8,0.966,245.0,287.0,None
8,RB,James Cook,BUF,15.1,117.3,250.4,0.970,232.0,262.0,Q4 best
9,WR,Amon-Ra St. Brown,DET,13.1,115.3,253.6,0.981,256.0,266.0,None


## The 1.01: Bijan, Gibbs, or Allen?

The board prices players one at a time, which cannot settle this — the question is what each choice
does to the *whole roster*. So each candidate is forced at the top pick and the draft is played out,
every candidate through the identical sampled rooms so the comparison is paired.

In [5]:
from scipy import stats
from src.gold.draft_plan import simulate_first_pick

first_pick = simulate_first_pick(LEAGUE, MY_SEAT, {
    "Josh Allen":     ["QB", "QB", "RB", "RB", "TE"],
    "Jahmyr Gibbs":   ["RB", "QB", "QB", "RB", "TE"],
    "Bijan Robinson": ["RB", "QB", "QB", "RB", "TE"],
}, trials=600)

wide = first_pick.pivot(index="trial", columns="player_name", values="points_vs_field")
summary = pd.DataFrame({
    "points_vs_field": wide.mean().round(1),
    "vs_allen": (wide.sub(wide["Josh Allen"], axis=0)).mean().round(1),
    "beats_allen": (wide.gt(wide["Josh Allen"], axis=0)).mean().round(3),
}).sort_values("points_vs_field", ascending=False)
summary

,points_vs_field,vs_allen,beats_allen
player_name,,,
Jahmyr Gibbs,130.4,18.4,0.803
Bijan Robinson,120.6,8.7,0.655
Josh Allen,112.0,0.0,0.000


In [6]:
for candidate in ["Bijan Robinson", "Jahmyr Gibbs"]:
    _, p = stats.ttest_rel(wide[candidate], wide["Josh Allen"])
    print(f"{candidate:<16} vs Josh Allen: p = {p:.2g}")

Bijan Robinson   vs Josh Allen: p = 1.9e-14
Jahmyr Gibbs     vs Josh Allen: p = 2.3e-51


**Take the running back.** Both backs beat Allen and the p-values are emphatic, but read the size
of the gaps rather than their significance: Allen still wins about a fifth of rooms against Gibbs
and better than a third against Bijan, on a season of roughly 1700 points. Two effects the model
deliberately does *not* capture both lean his way: in-season replacement asymmetry (waiver running
backs appear all year, waiver quarterbacks do not) and a durability measure built only from
injured-reserve stints, which misses one- and two-game absences. Read this as a real but slim edge,
not a mandate.

**The gap between the two backs is not evidence.** Run from the same seat on the same plan, Bijan
and Gibbs produce rosters that differ only by which of them is on it, so this simulation can do
nothing but restate their consensus projection gap — it would report the same margin if that
projection were badly wrong. `first_pick.ipynb` takes the question apart properly: the projection's
whole Gibbs-over-Bijan margin rests on a 2025 touchdown rate that does not persist, while the
in-house model, snap share, target share, offensive line and durability all favour Bijan. Treat
them as level.

## The plan is worth more than the player

Every opening composition, drafted 300 times from seat 1 against a resampled room:

In [7]:
plans = q('''
    SELECT plan, round(starter_points, 1) AS points, round(points_vs_field, 1) AS vs_field,
           round(finish_rank, 2) AS avg_finish, round(win_rate, 3) AS win_rate,
           round(top_third_rate, 3) AS top_third
    FROM draft_plans
    WHERE league_key = ? AND draft_slot = ?
    ORDER BY points_vs_field DESC
''', [LEAGUE, MY_SEAT])

pd.concat([plans.head(8), plans[plans.plan == "best available"], plans.tail(4)])

,plan,points,vs_field,avg_finish,win_rate,top_third
0,2QB2RB1TE,1696.9,116.9,2.61,0.363,0.863
1,2QB2RB1WR,1697.3,115.9,2.53,0.370,0.870
2,2QB3RB,1682.8,103.2,3.15,0.300,0.777
3,2QB1RB1WR1TE,1684.4,102.6,2.99,0.313,0.817
4,1QB2RB1WR1TE,1674.5,92.2,3.80,0.317,0.657
5,2QB1RB2TE,1666.9,85.6,3.56,0.217,0.720
6,2QB1RB2WR,1668.5,85.0,3.68,0.207,0.690
7,1QB3RB1WR,1666.7,84.6,3.99,0.277,0.657
12,best available,1633.4,50.8,5.31,0.183,0.527
33,1RB4WR,1502.3,-83.8,10.33,0.020,0.117


**Every one of the best openings takes two quarterbacks.** Drafting pure best-available finishes
13th of 37 plans — choosing a plan at all is worth about 64 points, against the roughly 14 that
separate the best of these three players at 1.01 from the worst. The plan is several times the
decision.

Best single ordering in the sweep: **RB, QB, QB, RB, TE**.

## Who actually reaches each of your picks

ADP is a mean and its standard deviation is the spread of real draft positions, so "still there at
pick k" is a tail probability rather than a guess. These are marginal probabilities — one player at
a time — so they do not add up across a position.

In [8]:
for pick in [1, 28, 29, 56, 57]:
    print(f"\n=== pick {pick} ===")
    print(q('''
        SELECT position AS pos, player_name, consensus_adp AS adp,
               round(points_over_replacement, 1) AS por, round(p_available, 2) AS p
        FROM draft_availability
        WHERE league_key = ? AND draft_slot = ? AND overall_pick = ? AND p_available >= 0.15
        ORDER BY points_over_replacement DESC
        LIMIT 6
    ''', [LEAGUE, MY_SEAT, pick]).to_string(index=False))


=== pick 1 ===
pos     player_name  adp   por   p
 RB    Jahmyr Gibbs  1.8 181.7 1.0
 RB  Bijan Robinson  2.6 172.2 1.0
 QB      Josh Allen  1.8 145.7 1.0
 RB Jonathan Taylor  5.5 132.4 1.0
 WR   Ja'Marr Chase  6.4 130.0 1.0
 WR      Puka Nacua  4.7 128.1 1.0

=== pick 28 ===
pos        player_name  adp  por    p
 RB     Saquon Barkley 24.5 94.4 0.29
 WR        CeeDee Lamb 25.1 94.4 0.33
 RB        Chase Brown 29.7 92.8 0.63
 WR       Drake London 24.3 92.1 0.25
 RB      Ashton Jeanty 30.8 89.5 0.66
 RB Kenneth Walker III 37.6 88.9 0.90

=== pick 29 ===
pos        player_name  adp  por    p
 RB     Saquon Barkley 24.5 94.4 0.23
 WR        CeeDee Lamb 25.1 94.4 0.26
 RB        Chase Brown 29.7 92.8 0.57
 WR       Drake London 24.3 92.1 0.19
 RB      Ashton Jeanty 30.8 89.5 0.61
 RB Kenneth Walker III 37.6 88.9 0.88

=== pick 56 ===
pos    player_name  adp  por    p
 TE   Trey McBride 64.3 76.9 0.77
 RB Travis Etienne 57.0 66.7 0.58
 RB  D'Andre Swift 58.5 66.0 0.66
 QB   Tyler Shough 4

## The quarterback cliff

14 quarterbacks are gone by pick 28. This is what makes the second one urgent and why mock drafts
against autopick bots mislead — bots draft off general rankings and let quarterbacks fall a dozen
picks past their real superflex price.

In [9]:
q('''
    SELECT count(*) FILTER (WHERE consensus_adp <= 28) AS gone_by_28,
           count(*) FILTER (WHERE consensus_adp <= 57) AS gone_by_57,
           count(*) FILTER (WHERE projected_points_adjusted >= 250) AS starter_quality,
           count(*) AS priced
    FROM draft_board WHERE league_key = ? AND position = 'QB'
''', [LEAGUE])

,gone_by_28,gone_by_57,starter_quality,priced
0,14,19,22,118


## Running backs behind a bad offensive line

Backs drafted in the ADP top 100 behind a bottom-quartile line returned **-6.6** points of surplus
against their draft price and beat that price only 35% of the time. Behind a top-quartile line:
**+20.3** and 56%. A 27-point swing at p=0.0055, positive in 7 of 7 seasons — and measured against
ADP, so the market is not already charging for it.

It applies to running backs and nobody else. Against the same grade, quarterbacks score -0.147,
tight ends +0.113 and receivers -0.048 — all noise.

Left as a warning rather than folded into the ranking: the relationship is not monotonic (the third
quartile edges the fourth), so it identifies backs to be wary of rather than backs to chase.

In [10]:
q('''
    SELECT player_name, team, consensus_adp AS adp, round(ol_grade, 1) AS ol_grade,
           round(points_over_replacement, 1) AS por, round(availability, 3) AS avail
    FROM draft_board
    WHERE league_key = ? AND ol_tier = 'Q1 worst' AND consensus_adp <= 130
    ORDER BY consensus_adp
''', [LEAGUE])

,player_name,team,adp,ol_grade,por,avail
0,Ashton Jeanty,LV,30.8,21.0,89.5,0.950
1,Josh Jacobs,GB,34.8,29.0,51.8,0.978
2,Kenneth Walker III,KC,37.6,16.1,88.9,0.955
3,Omarion Hampton,LAC,38.8,9.7,51.0,0.820
4,Cam Skattebo,NYG,52.8,24.2,26.5,0.773
5,Bhayshul Tuten,JAX,66.5,35.5,37.2,0.950
6,Quinshon Judkins,CLE,67.0,1.6,42.9,0.911
7,Jordan Mason,MIN,119.0,24.2,-7.7,0.889
8,Aaron Jones,MIN,124.5,24.2,-8.1,0.945


## Late rounds: chase the shallow positions

The instinct to keep taking receivers is wrong here, and replacement level says why. This league
starts 39 receivers, so the pool is 39 deep and the 40th is free. It starts 14 tight ends and 14
defenses. Value lives where the pool is shallow.

In [11]:
for pick in [112, 140, 168]:
    print(f"\n=== best available by position at pick {pick} (P >= 0.5) ===")
    print(q('''
        SELECT position AS pos, player_name, consensus_adp AS adp,
               round(points_over_replacement, 1) AS por, round(p_available, 2) AS p
        FROM draft_availability
        WHERE league_key = ? AND draft_slot = ? AND overall_pick = ? AND p_available >= 0.5
        QUALIFY row_number() OVER (PARTITION BY position ORDER BY points_over_replacement DESC) = 1
        ORDER BY por DESC
    ''', [LEAGUE, MY_SEAT, pick]).to_string(index=False))


=== best available by position at pick 112 (P >= 0.5) ===
pos    player_name   adp  por    p
DST            HOU 117.2 40.2 0.72
 TE   Travis Kelce 138.8 32.2 0.95
  K Brandon Aubrey 126.9 21.5 0.87
 QB     Geno Smith 120.8 13.4 0.70
 RB  Rachaad White 129.2 -0.5 0.86
 WR  KC Concepcion 135.6 -5.5 0.98

=== best available by position at pick 140 (P >= 0.5) ===
pos     player_name   adp    por    p
DST             BAL 154.7   24.7 0.83
  K      Jake Bates 144.6    6.7 0.73
 TE  Dalton Kincaid 142.5    4.5 0.57
 WR   Denzel Boston 158.0  -14.4 0.91
 RB     Woody Marks 148.5  -28.4 0.68
 QB Shedeur Sanders 139.5 -118.7 0.50

=== best available by position at pick 168 (P >= 0.5) ===
pos        player_name   adp   por    p
 WR    Adonai Mitchell 168.3 -44.1 0.53
 TE Darnell Washington 170.4 -64.1 0.59


Late receivers price out *below replacement* — they are worth less than what waivers will hand you
for nothing.

## Kicker last. Always.

The board says the best kicker is worth about 21 points over the last startable one. That number is
only spendable if you can identify him in August, and you cannot:

In [12]:
repeatability = q('''
    WITH kickers AS (
        SELECT player_id, season, sum(fantasy_points) AS points
        FROM weekly_stats
        WHERE position = 'K' AND season_type = 'REG' AND season BETWEEN 2015 AND 2025
        GROUP BY 1, 2 HAVING count(DISTINCT week) >= 12
    ),
    skill AS (
        SELECT player_id, position, season, sum(fantasy_points_ppr) AS points
        FROM weekly_stats
        WHERE position IN ('QB','RB','WR','TE') AND season_type = 'REG'
          AND season BETWEEN 2015 AND 2025
        GROUP BY 1, 2, 3 HAVING count(DISTINCT week) >= 12
    ),
    pairs AS (
        SELECT 'K' AS position, a.points AS this_year, b.points AS next_year
        FROM kickers a JOIN kickers b ON b.player_id = a.player_id AND b.season = a.season + 1
        UNION ALL
        SELECT a.position, a.points, b.points
        FROM skill a JOIN skill b
          ON b.player_id = a.player_id AND b.position = a.position AND b.season = a.season + 1
    )
    SELECT position, count(*) AS season_pairs, round(corr(this_year, next_year), 3) AS repeatability
    FROM pairs GROUP BY 1 ORDER BY repeatability DESC
''')
repeatability

,position,season_pairs,repeatability
0,WR,687,0.737
1,TE,293,0.699
2,RB,399,0.635
3,QB,179,0.368
4,K,221,-0.011


**-0.011.** Kicker scoring is not repeatable at all, so the top of the position is unpickable and
the pick belongs at 197. Defenses are modestly repeatable (points allowed carries year to year at
about +0.32), which argues for taking one in the last two or three rounds rather than dead last —
but the projection sources disagree with each other by more than the entire gap between the best
and worst startable defense, so do not reach.

## Bye weeks

A superflex slot accepts RB/WR/TE, so a quarterback bye does not require a third quarterback — it
requires a startable body. Check collisions before the last few picks, not after.

In [13]:
q('''
    SELECT f.bye, count(*) AS players,
           string_agg(b.player_name || ' (' || b.position || ')', ', '
                      ORDER BY b.points_over_replacement DESC) AS who
    FROM draft_board b
    JOIN (SELECT name, position, any_value(bye) AS bye FROM ffc_adp
          WHERE scoring_format = '2qb' AND season = 2026 GROUP BY 1, 2) f
      ON f.name = b.player_name AND f.position = b.position
    WHERE b.league_key = ? AND b.consensus_adp <= 60
    -- f.bye breaks ties: several bye weeks hold the same number of players, and an unstable
    -- sort there makes a no-op re-run produce a diff.
    GROUP BY f.bye ORDER BY players DESC, f.bye
''', [LEAGUE])

,bye,players,who
0,6.0,11,"Jahmyr Gibbs (RB), Ja'Marr Chase (WR), De'Von ..."
1,11.0,11,"Bijan Robinson (RB), Puka Nacua (WR), Jaxon Sm..."
2,8.0,8,"Christian McCaffrey (RB), Brock Purdy (QB), Ja..."
3,10.0,8,"Saquon Barkley (RB), Jalen Hurts (QB), Bo Nix ..."
4,13.0,7,"Jonathan Taylor (RB), Derrick Henry (RB), Lama..."
5,7.0,5,"Josh Allen (QB), Jayden Daniels (QB), Justin H..."
6,14.0,5,"CeeDee Lamb (WR), Jeremiyah Love (RB), George ..."
7,5.0,3,"Patrick Mahomes (QB), Tetairoa McMillan (WR), ..."


## The card

1. **1.01 — Bijan or Gibbs.** Allen is defensible and costs about 8 points.
2. **28 / 29 — two quarterbacks**, unless one of the top backs or receivers has fallen a long way.
   Realistic tier: Herbert, Dart, Nix. Not Hurts, not Dak — they will be gone.
3. **56 / 57 — running back and tight end.** Check availability above rather than a ranking.
4. **84 through 141 — shallow positions.** Tight end and defense outprice late receivers.
5. **196 / 197 — kicker last, always.**
6. Avoid the flagged backs unless the price has genuinely fallen to you.